# Cox process
Want to reproduce [this](https://tinygp.readthedocs.io/en/stable/tutorials/likelihoods.html) using BlackJAX only, no numpyro.

## TODO remove

In [ ]:
arr = jnp.arange(32.0).reshape(4, 8)
mesh = jax.make_mesh((2, 4), ("x", "y"))
sharding = jax.sharding.NamedSharding(mesh, P("x", "y"))
print(sharding)
arr_sharded = jax.device_put(arr, sharding)

print(arr_sharded)
for shard in arr_sharded.addressable_shards:
    print(shard.device, shard.index)

result = jnp.sum(arr_sharded, axis=0)
print(result)
for shard in result.addressable_shards:
    print(shard.device, shard.index)

In [ ]:
import blackjax

num_chains = 8

inv_mass_matrix = np.array([0.5, 0.01])
step_size = 1e-3


loc, scale = 10, 20
observed = np.random.normal(loc, scale, size=1_000)


def logdensity_fn(loc, log_scale, observed=observed):
    """Univariate Normal"""
    scale = jnp.exp(log_scale)
    logjac = log_scale
    logpdf = jax.scipy.stats.norm.logpdf(observed, loc, scale)
    return logjac + jnp.sum(logpdf)


def logdensity(x):
    return logdensity_fn(**x)


from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

mesh = jax.make_mesh((num_chains,), ("chain",))
sharding = NamedSharding(mesh, P("chain"))

rng_key, sample_key = jax.random.split(rng_key)
sample_keys = jax.device_put(jax.random.split(sample_key, num_chains), sharding)
for shard in sample_keys.addressable_shards:
    print(shard.device, shard.index)


nuts = blackjax.nuts(logdensity, step_size, inv_mass_matrix)
initial_positions = {"loc": np.ones(MCMC_CHAINS), "log_scale": np.ones(MCMC_CHAINS)}
initial_states = jax.vmap(nuts.init, in_axes=(0))(initial_positions)
initial_states_sharded = jax.device_put(initial_states, sharding)
for shard in initial_states_sharded.position["loc"].addressable_shards:
    print(shard.device, shard.index)

In [ ]:
initial_states_sharded

## BlackJAX version

In [ ]:
import datetime

start = datetime.datetime.now()

import multiprocessing
import matplotlib.pyplot as plt
import numpy as np
from typing import NamedTuple
from functools import partial

import jax
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P
import jax.numpy as jnp
import blackjax
from jaxtyping import Float
from tinygp import GaussianProcess, kernels

CPU_AVAIL = multiprocessing.cpu_count()
jax.config.update("jax_num_cpu_devices", CPU_AVAIL)
jax.config.update("jax_enable_x64", True)

rnd_state = int(datetime.date.today().strftime("%Y%m%d"))
rng_key = jax.random.key(rnd_state)
SIGMA = 5  # TODO check
T = 20  # nb of datapoints
BURNIN = 5_000
MCMC_CHAINS = CPU_AVAIL

print(start.strftime("%Y-%m-%d %H:%M:%S"))
assert len(jax.devices()) == CPU_AVAIL

random = np.random.default_rng(rnd_state)
x = np.linspace(-3, 3, T)
true_log_rate = 2 * np.cos(2 * x)
y = random.poisson(np.exp(true_log_rate))


class Params(NamedTuple):
    u: Float[jax.Array, ""]
    m: Float[jax.Array, "{T}"]


def init_param(key, t):
    key1, key2 = jax.random.split(key, 2)
    u = jax.random.uniform(key1)

    kernel = SIGMA**2 * kernels.Matern52(jnp.exp(u))
    gp = GaussianProcess(kernel, t, diag=1e-5)
    # array of shape T
    m = gp.sample(key2)

    return Params(
        u=u,
        m=m,
    )


rng_key, keys_init = jax.random.split(rng_key)
initial_params = init_param(keys_init, x)
initial_params


@jax.jit
def logdensity(t, y, params):
    # hyperprior
    log_hyperprior = jax.scipy.stats.norm.logpdf(params.u)

    # (latent) prior
    kernel = SIGMA**2 * kernels.Matern52(jnp.exp(params.u))
    gp = GaussianProcess(kernel, t, diag=1e-5)
    log_prior = gp.log_probability(params.m)

    # likelihood
    log_likelihood = jnp.sum(jax.scipy.stats.poisson.logpmf(y, jnp.exp(params.m)))

    return log_hyperprior + log_prior + log_likelihood


warmup = blackjax.window_adaptation(blackjax.nuts, partial(logdensity, x, y))
rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)
(state, parameters_nuts), _ = warmup.run(warmup_key, initial_params, num_steps=10_000)
# use obtained parameters to define a new NUTS instance
nuts_adapted = blackjax.nuts(partial(logdensity, x, y), **parameters_nuts)
parameters_nuts


def run_one_chain(key, state, inference_algorithm, num_samples):
    # sharding with shard_map gives (1, D) per shard but
    # run_inference_algo requires (D, ) so squeezing is required
    key = jnp.squeeze(key, axis=0)
    state = jax.tree.map(lambda x: jnp.squeeze(x, axis=0), state)
    _, (samples, info) = blackjax.util.run_inference_algorithm(
        key,
        inference_algorithm,
        num_samples,
        initial_state=state,
    )
    return samples, info


mesh = jax.make_mesh((MCMC_CHAINS,), ("chain",))
sharding = NamedSharding(mesh, P("chain"))

rng_key, sample_key = jax.random.split(rng_key)
# need to shard: 1. keys and 2. initial conditions
# 1. KEYS
sample_keys = jax.device_put(jax.random.split(sample_key, MCMC_CHAINS), sharding)
for shard in sample_keys.addressable_shards:
    print(shard.device, shard.index)
# 2. INITIAL CONDITIONS
# convoluted things to just repeat arrays MCMC_CHAINS times for sharding later
initial_params_repeated = jax.tree_util.tree_map(
    lambda e: jnp.broadcast_to(
        e, (MCMC_CHAINS, e.shape[0]) if not jax.numpy.isscalar(e) else (MCMC_CHAINS)
    ),
    initial_params,
)
initial_states = jax.vmap(nuts_adapted.init, in_axes=(0))(initial_params_repeated)
initial_states = jax.tree_map(
    lambda x: jnp.repeat(x[None], MCMC_CHAINS, axis=0),
    nuts_adapted.init(initial_params),
)
initial_states_sharded = jax.device_put(initial_states, sharding)
for shard in initial_states_sharded.position.u.addressable_shards:
    print(shard.device, shard.index)

run_one_chain_bound = partial(
    run_one_chain,
    inference_algorithm=nuts_adapted,
    num_samples=1_000 + BURNIN,
)

sharded_states, history = jax.jit(
    jax.shard_map(
        run_one_chain_bound,
        mesh=mesh,
        in_specs=(P("chain"), P("chain")),  # key, state
        out_specs=P("chain"),
        check_vma=False,
    )
)(sample_keys, initial_states_sharded)

m_samples = sharded_states.position.m.reshape(
    MCMC_CHAINS, 1_000 + BURNIN, sharded_states.position.m.shape[-1]
)[:, BURNIN:]
m_samples

In [ ]:
# diagnostics see https://arviz-devs.github.io/EABM/Chapters/MCMC_diagnostics.html
#  1. R statistics
#  2. E-BFMI https://python.arviz.org/projects/stats/en/stable/api/generated/arviz_stats.bfmi.html
#  3. trace plots
#  4. rank plots
#  5. Divergent transitions
#  6. Low effective sample size (ESS)
# see also arviz_stats.diagnose

In [ ]:
q = np.percentile(m_samples, [5, 25, 50, 75, 95], axis=0)
plt.plot(x, np.exp(q[2]), color="C0", label="MCMC inferred rate")
plt.fill_between(x, np.exp(q[0]), np.exp(q[-1]), alpha=0.3, lw=0, color="C0")
plt.fill_between(x, np.exp(q[1]), np.exp(q[-2]), alpha=0.3, lw=0, color="C0")
plt.plot(x, np.exp(true_log_rate), "--", color="C1", label="true rate")
plt.plot(x, y, ".k", label="data")
plt.legend(loc=2)
plt.xlabel("x")
_ = plt.ylabel("counts")

## Numpyro version

In [ ]:
%%time

import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

from tinygp import GaussianProcess, kernels

jax.config.update("jax_enable_x64", True)
numpyro.set_host_device_count(4)

random = np.random.default_rng(203618)
x = np.linspace(-3, 3, 20)
true_log_rate = 2 * np.cos(2 * x)
y = random.poisson(np.exp(true_log_rate))


def model(x, y=None):
    # The parameters of the GP model
    # mean = numpyro.sample("mean", dist.Normal(20, 0.01))
    mean = numpyro.sample("mean", dist.Normal(0, 2))
    sigma = numpyro.sample("sigma", dist.HalfNormal(3.0))
    rho = numpyro.sample("rho", dist.HalfNormal(10.0))

    # Set up the kernel and GP objects
    kernel = sigma**2 * kernels.Matern52(rho)
    gp = GaussianProcess(kernel, x, diag=1e-5, mean=mean)

    # This parameter has shape (num_data,) and it encodes our beliefs about
    # the process rate in each bin
    log_rate = numpyro.sample("log_rate", gp.numpyro_dist())

    # Finally, our observation model is Poisson
    numpyro.sample("obs", dist.Poisson(jnp.exp(log_rate)), obs=y)


# Run the MCMC
nuts_kernel = numpyro.infer.NUTS(model, target_accept_prob=0.9)
mcmc = numpyro.infer.MCMC(
    nuts_kernel,
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=False,
)
rng_key = jax.random.PRNGKey(55873)
mcmc.run(rng_key, x, y=y)
samples = mcmc.get_samples()

q = np.percentile(samples["log_rate"], [5, 25, 50, 75, 95], axis=0)
plt.plot(x, np.exp(q[2]), color="C0", label="MCMC inferred rate")
plt.fill_between(x, np.exp(q[0]), np.exp(q[-1]), alpha=0.3, lw=0, color="C0")
plt.fill_between(x, np.exp(q[1]), np.exp(q[-2]), alpha=0.3, lw=0, color="C0")
plt.plot(x, np.exp(true_log_rate), "--", color="C1", label="true rate")
plt.plot(x, y, ".k", label="data")
plt.legend(loc=2)
plt.xlabel("x")
_ = plt.ylabel("counts")

In [ ]:
print(f"Total elasped time for the notebook: {(datetime.datetime.now() - start)}")